### Implémentation du notebook pour l’Indice de Sécheresse (SPEI simplifié)
Vous trouverez ci-dessous la marche à suivre détaillée pour construire un notebook complet dédié à la prédiction du SPEI simplifié (Standardized Precipitation-Evapotranspiration Index). L’objectif est de prévoir le déficit hydrique à partir des données météo journalières

---

1. Import des bibliothèques
2. Chargement et exploration des données
3. Feature engineering – calcul de la cible SPEI
4. Création des features explicatives (temporelles, rolling, lags)
5. Préparation des données pour la régression (split temporel, scaling)
6. Entraînement de modèles (baseline, Random Forest, XGBoost)
7. Sélection et sauvegarde du meilleur modèle
8. Interprétabilité (SHAP, importance des features)
9. Conclusion et export

### **1-importations des bibliotheques**

In [1]:
print("test")

test


In [ ]:
import pandas as pd
import numpy as np # type: ignore
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import shap
from pathlib import Path


DATASET_PATH = Path("C:/Users/LENOVO/Desktop/hackathon/HACKATHON-INDABAX-CAMEROON-2026-main/data/Dataset_complet_Meteo.csv")

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: DLL load failed while importing _multiarray_umath: Le module spécifié est introuvable.

ImportError: numpy._core.multiarray failed to import

### **2-Chargement et exploration**

In [ ]:
# Chargement (adapter le chemin)
df = pd.read_csv(DATASET_PATH, parse_dates=['time'], delimiter=";")
df.set_index('time', inplace=True)
df.sort_index(inplace=True)

# Aperçu
print(df.shape)
df.head()

# Vérifier les variables nécessaires
required_cols = ['precipitation_sum', 'et0_fao_evapotranspiration', 'city']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Colonne manquante : {col}")

# Analyse exploratoire
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
df.groupby('city')['precipitation_sum'].mean().sort_values().plot(kind='barh', ax=axes[0], title='Précipitations moyennes par ville')
df.groupby('city')['et0_fao_evapotranspiration'].mean().sort_values().plot(kind='barh', ax=axes[1], title='ET0 moyenne par ville')
plt.tight_layout()
plt.show()

NameError: name 'pd' is not defined

### **3 Feature engineering – Calcul de la cible SPEI**

In [ ]:
# Calcul du bilan hydrique journalier
df['water_balance'] = df['precipitation_sum'] - df['et0_fao_evapotranspiration']

# Grouper par ville pour calculer rolling statistics
def compute_spei(group, window=30):
    group['wb_roll_mean'] = group['water_balance'].rolling(window, min_periods=window).mean()
    group['wb_roll_std'] = group['water_balance'].rolling(window, min_periods=window).std()
    group['SPEI'] = (group['water_balance'] - group['wb_roll_mean']) / group['wb_roll_std']
    return group

df = df.groupby('city', group_keys=False).apply(compute_spei)

# Supprimer les lignes avec des NaN (premières fenêtres)
df.dropna(subset=['SPEI'], inplace=True)

Remarque : On peut aussi agréger par mois (plus simple mais moins fin). Si l’on souhaite une cible mensuelle, regrouper par (city, year, month) et sommer P et ET0, puis calculer le SPEI sur l’historique mensuel.

### **4 Features explicatives (entrées du modèle)**

On utilise les variables météo brutes et des rolling pour capturer l’état antérieur.

In [ ]:
# Features temporelles
df['month'] = df.index.month
df['day_of_year'] = df.index.dayofyear
df['sin_doy'] = np.sin(2 * np.pi * df['day_of_year'] / 365)
df['cos_doy'] = np.cos(2 * np.pi * df['day_of_year'] / 365)

# Rolling sur les précipitations et ET0 (pour informer le modèle)
for var in ['precipitation_sum', 'et0_fao_evapotranspiration', 'water_balance']:
    for window in [7, 14, 30]:
        df[f'{var}_roll{window}'] = df.groupby('city')[var].transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )

# Lags du water_balance (optionnel, si prédiction à J+1)
for lag in [1, 2, 3]:
    df[f'wb_lag{lag}'] = df.groupby('city')['water_balance'].shift(lag)

# Sélection des features (ajuster selon disponibilité)
feature_cols = [
    'precipitation_sum', 'et0_fao_evapotranspiration', 'water_balance',
    'precipitation_sum_roll7', 'et0_fao_evapotranspiration_roll7', 'water_balance_roll7',
    'precipitation_sum_roll14', 'et0_fao_evapotranspiration_roll14', 'water_balance_roll14',
    'precipitation_sum_roll30', 'et0_fao_evapotranspiration_roll30', 'water_balance_roll30',
    'month', 'sin_doy', 'cos_doy'
]
# Ajouter d'autres variables météo si disponibles : temperature_2m_mean, wind_speed_10m_max, shortwave_radiation_sum
# (Elles améliorent la prédiction de l'ET0 et des précipitations)

X = df[feature_cols].copy()
y = df['SPEI']

# Supprimer les lignes avec NaN (à cause des lags)
X.dropna(inplace=True)
y = y.loc[X.index]

### **5 Split temporel (pas de fuite future)**

# Séparation avant/après 2024 (train 2020-2023, test 2024-2025)
train_mask = X.index.year < 2024
test_mask = X.index.year >= 2024

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f"Train : {X_train.shape}, Test : {X_test.shape}")

### **6 Normalisation**

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### **7 Entraînement et comparaison de modèles**

models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    results[name] = {'MAE': mae, 'R2': r2}
    print(f"{name}: MAE={mae:.3f}, R2={r2:.3f}")

# Meilleur modèle (ex: XGBoost)
best_model = models['XGBoost']

### **8 Sauvegarde du meilleur modèle et du scaler**

In [ ]:
joblib.dump(best_model, 'best_spei_model.pkl')
joblib.dump(scaler, 'scaler_spei.pkl')
# Sauvegarder aussi la liste des features pour cohérence
with open('spei_features.txt', 'w') as f:
    f.write('\n'.join(feature_cols))

### **9 Interprétabilité**

Importance des features (pour RandomForest/XGBoost) :

In [ ]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)
print(feature_importance.head(10))

# Graphique
plt.figure(figsize=(10,6))
sns.barplot(data=feature_importance.head(10), x='importance', y='feature')
plt.title('Top 10 des features pour la prédiction du SPEI')
plt.show()

### **SHAP (exemple sur un échantillon de test)**

In [ ]:
# Pour XGBoost
explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_test_scaled[:500])  # 500 premiers échantillons
shap.summary_plot(shap_values, X_test_scaled[:500], feature_names=feature_cols)

# Force plot pour une prédiction individuelle
shap.initjs()
shap.force_plot(explainer.expected_value, shap_values[0, :], X_test_scaled[0, :], feature_names=feature_cols)

### **10 Visualisation des prédictions**

In [ ]:
# Comparaison sur la période de test pour une ville choisie
test_city = 'Yaounde'  # à adapter
mask_city = X_test.index.get_level_values('city') == test_city  # si city est dans l'index
# Sinon, il faut récupérer city depuis df_original
# Plus simple : récupérer les index de X_test, puis filtrer df original
y_test_city = y_test.loc[X_test.index[X_test.index.get_level_values('city') == test_city]]
y_pred_city = y_pred[X_test.index.get_level_values('city') == test_city]

plt.figure(figsize=(12,4))
plt.plot(y_test_city.index, y_test_city, label='Vrai SPEI')
plt.plot(y_test_city.index, y_pred_city, label='Prédit SPEI')
plt.title(f'SPEI - {test_city} (période test)')
plt.legend()
plt.show()

 ### **Points spécifiques au dataset**
Vérification de et0_fao_evapotranspiration : si la colonne n’existe pas, il faudra la calculer via la formule de Penman-Monteith FAO (nécessite température, vent, humidité, rayonnement). Dans le document, elle est listée comme disponible.

Gestion des valeurs manquantes : avant de calculer les rolling, interpoler par ville (ex: df.groupby('city')['precipitation_sum'].apply(lambda x: x.interpolate())).

Fréquence : le SPEI est souvent utilisé sur des échelles de 1, 3, 6 mois. Pour un hackathon, l’échelle de 30 jours est raisonnable. Vous pouvez aussi proposer une version mensuelle (regrouper par mois, puis calculer le SPEI sur la série mensuelle).

###  **Améliorations possibles**
Optimisation des hyperparamètres avec GridSearchCV et TimeSeriesSplit.

Prédiction multi‑sorties : prévoir simultanément SPEI à plusieurs horizons (1 semaine, 1 mois).

Intégration d’un modèle LSTM si les séquences temporelles longues sont exploitables (nécessite davantage de données)